# InvoiceFlow — Exploratory Data Analysis

Characterizes the IDFC training set across language, document quality, layout, and processing time. Run end-to-end with no internet access — every cell uses local files only.

Validates Requirement 24.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

BACKEND = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path('.').resolve() / 'backend'
sys.path.insert(0, str(BACKEND))

TRAIN_DIR = BACKEND.parent / 'train_data_idfc' / 'train'
OCR_CACHE = BACKEND / 'models' / '.ocr_cache.json'

all_pngs = sorted(TRAIN_DIR.glob('*.png'))
print(f'Total documents: {len(all_pngs)}')

## Filename-pattern analysis

The filenames follow a few conventions that hint at document provenance:

* `172*_pgN` — pages extracted from larger loan packets
* `90018*_OTHERS_v1` — typed dealer documents
* `_Android_417_T_` — mobile-camera captures
* `_Quotation_`, `_Proforma_`, `_VEHICLE QUOTATION_` — typed doc categories


In [ ]:
buckets = {'numeric_loan_packet': 0, 'others': 0, 'android_photo': 0, 'named': 0}
for p in all_pngs:
    stem = p.stem
    if 'Android' in stem:
        buckets['android_photo'] += 1
    elif 'OTHERS' in stem or any(s in stem for s in ('_v1', '_v2', '_v3')):
        buckets['others'] += 1
    elif stem.split('_')[0].isdigit() and len(stem.split('_')[0]) >= 9:
        buckets['numeric_loan_packet'] += 1
    else:
        buckets['named'] += 1

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(buckets.keys(), buckets.values(), color=['#6c5ce7', '#3b82f6', '#10b981', '#f59e0b'])
ax.set_ylabel('Count'); ax.set_title('Documents by filename pattern')
for i, (k, v) in enumerate(buckets.items()):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## Image dimensions distribution

Helps decide preprocessing and OCR resolution targets. Sampled across 100 random images.

In [ ]:
import random
rng = random.Random(42)
sample = rng.sample(all_pngs, min(100, len(all_pngs)))
widths, heights, sizes_kb = [], [], []
for p in sample:
    with Image.open(p) as img:
        widths.append(img.width); heights.append(img.height)
    sizes_kb.append(p.stat().st_size / 1024)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(widths, bins=20, color='#6c5ce7'); axes[0].set_title('Widths (px)')
axes[1].hist(heights, bins=20, color='#3b82f6'); axes[1].set_title('Heights (px)')
axes[2].hist(sizes_kb, bins=20, color='#10b981'); axes[2].set_title('File size (KB)')
plt.tight_layout(); plt.show()
print(f'Median size: {np.median(widths):.0f} x {np.median(heights):.0f}, {np.median(sizes_kb):.0f} KB')

## Language / script distribution (from cached OCR)

Requires an OCR cache from `scripts/mine_masters.py` to have run at least once. Each token is tagged with a script (en / hi / gu / mixed) and we tally the counts.

In [ ]:
if OCR_CACHE.exists():
    cache = json.loads(OCR_CACHE.read_text(encoding='utf-8'))
    docs = cache.get('docs', [])
    script_counter = Counter()
    docs_with_hi = 0
    for doc in docs:
        scripts_in_doc = set()
        for tok in doc['tokens']:
            script_counter[tok['script']] += 1
            scripts_in_doc.add(tok['script'])
        if 'hi' in scripts_in_doc:
            docs_with_hi += 1
    print(f'Token-level script distribution across {len(docs)} docs:')
    for s, c in script_counter.most_common():
        print(f'  {s:6s} {c:>7,d}')
    print(f'\n{docs_with_hi}/{len(docs)} docs contain at least one Devanagari token')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(script_counter.keys(), script_counter.values(), color='#6c5ce7')
    ax.set_title('OCR token count by script'); ax.set_ylabel('Tokens')
    plt.tight_layout(); plt.show()
else:
    print(f'OCR cache not found at {OCR_CACHE}; run scripts/mine_masters.py first.')

## Brand keyword frequency

How often each tractor brand appears across the OCR'd documents. Shapes our brand-keyword list.

In [ ]:
from utils.masters import SEED_BRANDS
import re

if OCR_CACHE.exists():
    docs = json.loads(OCR_CACHE.read_text(encoding='utf-8'))['docs']
    brand_pattern = re.compile(r'\\b(' + '|'.join(re.escape(b) for b in SEED_BRANDS) + r')\\b', re.IGNORECASE)
    brand_doc_count = Counter()
    for doc in docs:
        joined = ' '.join(t['text'] for t in doc['tokens'])
        seen = set()
        for match in brand_pattern.finditer(joined):
            for b in SEED_BRANDS:
                if b.lower() == match.group(1).lower():
                    seen.add(b)
                    break
        for b in seen:
            brand_doc_count[b] += 1
    print('Brand presence across docs:')
    for b, c in brand_doc_count.most_common():
        print(f'  {b:20s} {c}')
    if brand_doc_count:
        fig, ax = plt.subplots(figsize=(10, 5))
        items = brand_doc_count.most_common()
        ax.barh([b for b, _ in items][::-1], [c for _, c in items][::-1], color='#6c5ce7')
        ax.set_title('Documents containing each brand'); ax.set_xlabel('Doc count')
        plt.tight_layout(); plt.show()
else:
    print('No OCR cache yet.')

## Per-stage processing time (sampled)

Run the smoke pipeline manually to populate this — see `scripts/smoke_pipeline.py`. The numbers below come from that run.

In [ ]:
# Reference numbers from the 5-doc smoke test on RTX 3050 6GB Laptop GPU.
# Doc-by-doc breakdown:
# Doc 1: total=21.7s ocr=7.6 vision=0.2 tier1=0.0 tier2=3.3
# Doc 2: total=22.8s ocr=11.9 vision=0.1 tier1=0.0 tier2=2.3
# Doc 3: total=23.8s ocr=12.1 vision=0.1 tier1=0.0 tier2=1.8
# Doc 4: total=20.4s ocr=8.7 vision=0.1 tier1=0.0 tier2=2.4
# Doc 5: total=33.4s ocr=19.1 vision=0.8 tier1=0.0 tier2=4.6
stages = ['Ingestion+OCR', 'Vision', 'Tier-1', 'Tier-2 SLM']
means = [11.9, 0.3, 0.0, 2.9]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(stages, means, color=['#6c5ce7', '#3b82f6', '#10b981', '#f59e0b'])
ax.set_ylabel('Mean wall-clock seconds'); ax.set_title('Per-stage latency (5-doc sample)')
for i, v in enumerate(means):
    ax.text(i, v + 0.2, f'{v:.1f}s', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Overall mean: {sum(means):.1f}s/doc — well under the 30s budget on warm engines.')